# DashBot Ablation Study on Google Colab

Notebook nay dung de clone repo DashBot tu GitHub, cai dependencies, mount Google Drive, va chay 3 baseline con thieu: `DashBot-ind.`, `DashBot-pen.`, `DQN`.

Chay tung baseline mot. Dung runtime thi log/checkpoint van nam trong Google Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repo

Doi `YOUR_GITHUB_USERNAME` va `YOUR_REPO_NAME` thanh repo cua ban.

In [ ]:
!rm -rf /content/DashBot
!git clone https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME.git /content/DashBot
%cd /content/DashBot

## 3. Install dependencies

In [ ]:
!pip install -r requirements.txt

## 4. Prepare output folders

Ket qua se duoc luu vao Google Drive de khong mat khi Colab disconnect.

In [ ]:
import os, shutil
DRIVE_OUT = '/content/drive/MyDrive/dashbot_ablation'
os.makedirs(f'{DRIVE_OUT}/reports/ablation', exist_ok=True)
os.makedirs(f'{DRIVE_OUT}/weights/ablation', exist_ok=True)
os.makedirs('reports/ablation', exist_ok=True)
os.makedirs('backend/dashbot/weights/ablation', exist_ok=True)
print(DRIVE_OUT)

## 5. Reuse old DashBot log

Neu ban da push `reports/training_curve_a3c.csv` len GitHub, cell nay se copy no vao folder ablation de ve chung voi 3 baseline moi.

In [ ]:
from pathlib import Path
src = Path('reports/training_curve_a3c.csv')
dst = Path('reports/ablation/training_curve_dashbot.csv')
if src.exists():
    shutil.copyfile(src, dst)
    shutil.copyfile(dst, f'{DRIVE_OUT}/reports/ablation/training_curve_dashbot.csv')
    print('copied old DashBot log')
else:
    print('reports/training_curve_a3c.csv not found; train DashBot full separately if needed')

## 6. Train DashBot-ind.

Paper ablation: bo co che du doan tuan tu, cac classification heads doc lap.

In [ ]:
!python scripts/train_a3c.py --variant dashbot-ind --steps 500000 --workers 4 --rollout-length 50 --learning-rate 1e-4 --entropy-coef 0.01 --hidden-size 128 --log-interval 5000 --log-csv reports/ablation/training_curve_dashbot_ind.csv --checkpoint-interval 50000 --checkpoint-dir backend/dashbot/weights/ablation/checkpoints_dashbot_ind --save-path backend/dashbot/weights/ablation/dashbot_ind_actor_critic.pth
!cp reports/ablation/training_curve_dashbot_ind.csv {DRIVE_OUT}/reports/ablation/
!cp backend/dashbot/weights/ablation/dashbot_ind_actor_critic.pth {DRIVE_OUT}/weights/ablation/

## 7. Train DashBot-pen.

Paper ablation: thay constrained sampling bang penalty khi cau hinh khong hop le.

In [ ]:
!python scripts/train_a3c.py --variant dashbot-pen --steps 500000 --workers 4 --rollout-length 50 --learning-rate 1e-4 --entropy-coef 0.01 --hidden-size 128 --invalid-penalty -1.0 --log-interval 5000 --log-csv reports/ablation/training_curve_dashbot_pen.csv --checkpoint-interval 50000 --checkpoint-dir backend/dashbot/weights/ablation/checkpoints_dashbot_pen --save-path backend/dashbot/weights/ablation/dashbot_pen_actor_critic.pth
!cp reports/ablation/training_curve_dashbot_pen.csv {DRIVE_OUT}/reports/ablation/
!cp backend/dashbot/weights/ablation/dashbot_pen_actor_critic.pth {DRIVE_OUT}/weights/ablation/

## 8. Train DQN

Baseline thay A3C bang Deep Q-Network.

In [ ]:
!python scripts/train_dqn.py --steps 500000 --learning-rate 1e-4 --hidden-size 128 --batch-size 64 --target-update-interval 5000 --log-interval 5000 --log-csv reports/ablation/training_curve_dqn.csv --checkpoint-interval 50000 --checkpoint-dir backend/dashbot/weights/ablation/checkpoints_dqn --save-path backend/dashbot/weights/ablation/dashbot_dqn.pth
!cp reports/ablation/training_curve_dqn.csv {DRIVE_OUT}/reports/ablation/
!cp backend/dashbot/weights/ablation/dashbot_dqn.pth {DRIVE_OUT}/weights/ablation/

## 9. Plot Fig. 6-style ablation curve

In [ ]:
!python scripts/plot_paper_figures.py learning-curve --dashbot-log reports/ablation/training_curve_dashbot.csv --dashbot-ind-log reports/ablation/training_curve_dashbot_ind.csv --dashbot-pen-log reports/ablation/training_curve_dashbot_pen.csv --dqn-log reports/ablation/training_curve_dqn.csv --output reports/fig6_ablation_learning_curve.png
!cp reports/fig6_ablation_learning_curve.png {DRIVE_OUT}/reports/
from IPython.display import Image, display
display(Image('reports/fig6_ablation_learning_curve.png'))

## Quick smoke test commands

Neu chi muon test notebook co chay khong, doi `--steps 500000` thanh `--steps 100` va them `--checkpoint-interval 0`.